# Requisito 1 Ingesta

In [0]:
import os
import time
import zipfile
import requests

volume_path = "/Volumes/proyecto_datos/default/data/ais_data"
os.makedirs(volume_path, exist_ok=True)

urls = [
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_02.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_03.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_04.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_05.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_06.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_07.zip"
]

def download_with_retries(url, dest_path, max_retries=3, delay=5):
    """Downloads a file with automatic retry logic."""
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Downloading {os.path.basename(dest_path)} (Attempt {attempt}/{max_retries})...")
            response = requests.get(url, stream=True, timeout=60)
            response.raise_for_status()
            
            with open(dest_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(delay)
            else:
                raise RuntimeError(f"Failed to download {url} after {max_retries} attempts.")

def verify_zip_integrity(zip_filepath):
    """Verifies if the zip file is valid and not corrupted."""
    print(f"Verifying integrity of {os.path.basename(zip_filepath)}...")
    if not zipfile.is_zipfile(zip_filepath):
        return False
    
    with zipfile.ZipFile(zip_filepath, 'r') as zf:
        # testzip returns None if there are no bad files
        bad_file = zf.testzip()
        return bad_file is None

# Main execution logic
for url in urls:
    zip_name = url.split("/")[-1]
    zip_filepath = os.path.join(volume_path, zip_name)
    
    # 1. Download with retries
    download_with_retries(url, zip_filepath, max_retries=3)
    
    # 2. Verify integrity
    if not verify_zip_integrity(zip_filepath):
        raise ValueError(f"Integrity check failed for {zip_name}. The file is corrupted.")
    print(f"Integrity verified for {zip_name}.")
    
    # 3. Extract to Volume
    print(f"Extracting {zip_name}...")
    with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
        zip_ref.extractall(volume_path)
        
    os.remove(zip_filepath)

print("All files downloaded, verified, and extracted successfully.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "/Volumes/proyecto_datos/default/data/ais_data"

ais_schema = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length double,
    Width double,
    Draft double,
    Cargo string,
    TransceiverClass string
"""

df_ais = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(ais_schema)
    .csv(f"{BASE}/*.csv")
)

In [0]:
display(df_ais.limit(10))

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
367138690,2023-06-01T00:00:01.000Z,30.28335,-91.22283,0,273,511,ST FRANCISVILLE,null,WDD3968,60,0,43,18,null,60,A
368036960,2023-06-01T00:00:01.000Z,40.49818,-80.07596,3.3,118.6,131,GALE R RHODES,IMO8199788,WDK2590,0,12,20,7,2.6,0,A
368095340,2023-06-01T00:00:01.000Z,29.99678,-93.94959,0,309.8,511,GAMBLER,null,WDK8562,52,15,19,8,0,52,A
248301000,2023-06-01T00:00:03.000Z,26.13342,-80.11909,0,260.5,86,ATLAS,null,9HB5599,37,5,31,8,3,37,A
367186410,2023-06-01T00:00:03.000Z,30.00762,-93.95571,5.7,164,164,BRIAN O DANIELS,null,WDD7438,31,12,26,9,null,31,A
367108820,2023-06-01T00:00:05.000Z,37.82833,-76.27936,0.1,235.2,511,TIDELANDS,IMO7309443,WX5026,30,0,61,9,null,30,A
367320740,2023-06-01T00:00:05.000Z,38.73458,-82.87265,4.7,350.3,349,TENNESSEE,IMO8836584,WDE2124,31,0,42,12,null,52,A
368101220,2023-06-01T00:00:03.000Z,32.6922,-117.14808,0,360,511,WILLIAM F,null,WDK9183,33,15,40,18,0,33,A
367529860,2023-06-01T00:00:08.000Z,29.61437,-95.00393,0.2,197.6,285,C R MCCASKILL,null,WDG3808,90,3,67,null,null,33,A
367558580,2023-06-01T00:00:09.000Z,27.3326,-82.54605,0,233.2,221,DAYS LIKE THIS,null,WDG6586,37,0,40,7,null,37,A


In [0]:
df_ais.count()

60533559

# Requisito 2 Exploración y perfilamiento

In [0]:
# Add date column for daily breakdowns
df_prepared = df_ais.withColumn("EventDate", F.to_date("BaseDateTime"))

# Flag duplicate records (exact duplicates across all key AIS payload attributes)
duplicate_window = Window.partitionBy("MMSI", "BaseDateTime", "LAT", "LON").orderBy("BaseDateTime")
df_flagged = df_prepared.withColumn("row_num", F.row_number().over(duplicate_window))

In [0]:
display(df_flagged.limit(10))

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,EventDate,row_num
0,2023-06-01T11:53:17.000Z,41.66808,-70.05943,0,0,0,null,null,null,0,null,14,4,0,0,B,2023-06-01,1
0,2023-06-01T14:14:17.000Z,41.60365,-70.27104,9.7,303.8,303,null,null,null,0,null,14,4,2.1,37,B,2023-06-01,1
0,2023-06-01T15:46:59.000Z,41.6331,-70.27112,6.3,197.9,197,null,null,null,0,null,14,4,2.1,37,B,2023-06-01,1
0,2023-06-05T02:16:18.000Z,21.28996,-157.21604,6.5,78.7,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T02:23:24.000Z,21.29247,-157.20276,6.2,79.6,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T03:36:32.000Z,21.32243,-157.06323,6.2,72.5,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T05:51:19.000Z,21.37758,-156.80434,6.6,76.5,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T06:07:31.000Z,21.38265,-156.77219,6.6,82.6,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T08:43:13.000Z,21.42387,-156.45416,6.6,86.8,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1
0,2023-06-05T09:07:34.000Z,21.42788,-156.40383,7.1,85.9,511,null,null,null,0,null,14,4,2.1,37,B,2023-06-05,1


In [0]:
# 1. Total records and total unique vessels
total_records = df_ais.count()
total_unique_vessels = df_ais.select("MMSI").distinct().count()

print(f"Total AIS Records: {total_records:,}")
print(f"Total Unique MMSIs: {total_unique_vessels:,}")

# 2. Positions and Unique Vessels per Day
daily_summary = (
    df_flagged
    .groupBy("EventDate")
    .agg(
        F.count("*").alias("Total_Positions"),
        F.countDistinct("MMSI").alias("Unique_Vessels")
    )
    .orderBy("EventDate")
)

display(daily_summary)

Total AIS Records: 60,533,559
Total Unique MMSIs: 31,871


EventDate,Total_Positions,Unique_Vessels
2023-06-01,8808904,20448
2023-06-02,9052241,21153
2023-06-03,8036348,20505
2023-06-04,8522645,19720
2023-06-05,8613757,19615
2023-06-06,8587938,19717
2023-06-07,8911726,20086


In [0]:
# Distribution by Vessel Type (Top 15)
vessel_type_dist = (
    df_ais
    .groupBy("VesselType")
    .agg(
        F.count("*").alias("Position_Count"),
        F.countDistinct("MMSI").alias("Unique_Vessels"),
        F.round(F.avg("SOG"), 2).alias("Avg_SOG_Knots")
    )
    .orderBy(F.desc("Position_Count"))
)

display(vessel_type_dist)

# Physical Size Metrics Distribution (Length, Width, Draft)
size_metrics = df_ais.select(
    F.expr("percentile_approx(Length, array(0.25, 0.50, 0.75, 0.99))").alias("Length_p25_p50_p75_p99"),
    F.expr("percentile_approx(Width, array(0.25, 0.50, 0.75, 0.99))").alias("Width_p25_p50_p75_p99"),
    F.expr("percentile_approx(Draft, array(0.25, 0.50, 0.75, 0.99))").alias("Draft_p25_p50_p75_p99")
)

display(size_metrics)

VesselType,Position_Count,Unique_Vessels,Avg_SOG_Knots
31,16558120,3451,1.9
37,14476696,12934,1.64
60,4795209,1708,4.43
30,4010188,2192,2.47
36,3876264,4243,1.98
90,3848162,1532,2.03
70,3837926,1563,6.37
52,1921565,482,2.09
80,1789168,818,5.87
57,1323765,242,1.91


Length_p25_p50_p75_p99,Width_p25_p50_p75_p99,Draft_p25_p50_p75_p99
"List(15, 22, 38, 323)","List(5, 8, 11, 48)","List(2.4, 3.7, 7.3, 17)"


In [0]:
quality_diagnostics = df_flagged.select(
    # Total count for percentage calculations
    F.count("*").alias("total_rows"),

    # 1. Out-of-range positions (Valid LAT: -90 to 90, Valid LON: -180 to 180)
    F.sum(F.when((F.col("LAT") < -90) | (F.col("LAT") > 90) | (F.col("LAT") == 0.0), 1).otherwise(0)).alias("invalid_lat_count"),
    F.sum(F.when((F.col("LON") < -180) | (F.col("LON") > 180) | (F.col("LON") == 0.0), 1).otherwise(0)).alias("invalid_lon_count"),

    # 2. Speed over ground (SOG) anomalies (> 102 knots is max AIS standard; > 60 knots is physically impossible for commercial shipping)
    F.sum(F.when(F.col("SOG") > 60.0, 1).otherwise(0)).alias("impossible_speed_count"),
    F.sum(F.when(F.col("SOG") < 0.0, 1).otherwise(0)).alias("negative_speed_count"),

    # 3. Invalid / Anomalous MMSIs (MMSIs must be 9-digit identifiers; 000000000 or non-9-digit lengths are invalid)
    F.sum(F.when((F.length(F.col("MMSI")) != 9) | (F.col("MMSI") == "000000000") | F.col("MMSI").rlike("^0+$"), 1).otherwise(0)).alias("anomalous_mmsi_count"),

    # 4. Heading anomalies (Valid Heading: 0 to 359 degrees; 511 indicates "not available")
    F.sum(F.when(F.col("Heading") == 511.0, 1).otherwise(0)).alias("heading_not_available_count"),
    F.sum(F.when((F.col("Heading") < 0.0) | ((F.col("Heading") > 359.0) & (F.col("Heading") != 511.0)), 1).otherwise(0)).alias("invalid_heading_count"),

    # 5. Dimension anomalies (Length <= 0 or Width <= 0)
    F.sum(F.when((F.col("Length") <= 0) | (F.col("Width") <= 0), 1).otherwise(0)).alias("invalid_dimensions_count"),

    # 6. Duplicates
    F.sum(F.when(F.col("row_num") > 1, 1).otherwise(0)).alias("duplicate_records_count")
)

# Reshape into a readable summary table
diag_row = quality_diagnostics.collect()[0]
total = diag_row["total_rows"]

quality_summary = spark.createDataFrame([
    ("Invalid Latitude (|LAT| > 90 or 0.0)", diag_row["invalid_lat_count"], (diag_row["invalid_lat_count"] / total) * 100),
    ("Invalid Longitude (|LON| > 180 or 0.0)", diag_row["invalid_lon_count"], (diag_row["invalid_lon_count"] / total) * 100),
    ("Impossible Speed (SOG > 60 knots)", diag_row["impossible_speed_count"], (diag_row["impossible_speed_count"] / total) * 100),
    ("Negative Speed (SOG < 0)", diag_row["negative_speed_count"], (diag_row["negative_speed_count"] / total) * 100),
    ("Anomalous MMSI (Length != 9 or Dummy)", diag_row["anomalous_mmsi_count"], (diag_row["anomalous_mmsi_count"] / total) * 100),
    ("Heading Uninformative (511.0)", diag_row["heading_not_available_count"], (diag_row["heading_not_available_count"] / total) * 100),
    ("Invalid Heading Out of Bounds", diag_row["invalid_heading_count"], (diag_row["invalid_heading_count"] / total) * 100),
    ("Invalid Dimensions (Length/Width <= 0)", diag_row["invalid_dimensions_count"], (diag_row["invalid_dimensions_count"] / total) * 100),
    ("Duplicate Positions (MMSI + Timestamp + Coordinates)", diag_row["duplicate_records_count"], (diag_row["duplicate_records_count"] / total) * 100)
], ["Quality_Issue", "Affected_Rows", "Percentage_Of_Total"])

display(quality_summary)

Quality_Issue,Affected_Rows,Percentage_Of_Total
Invalid Latitude (|LAT| > 90 or 0.0),0,0
Invalid Longitude (|LON| > 180 or 0.0),0,0
Impossible Speed (SOG > 60 knots),160613,0.26532885667601336
Negative Speed (SOG < 0),0,0
Anomalous MMSI (Length != 9 or Dummy),49897,0.08242865746585294
Heading Uninformative (511.0),33535628,55.40005998986447
Invalid Heading Out of Bounds,1061,0.0017527467697711283
Invalid Dimensions (Length/Width <= 0),2284033,3.7731682024511395
Duplicate Positions (MMSI + Timestamp + Coordinates),1400,0.002312766708463317


## Diagnostico y Hallazgos Principales
1. Volumen General y Comportamiento Diario
+ Total de registros procesados: 60,533,559 posiciones.
+ Buques unicos en la semana: 31,871 MMSI distintos.
+ Actividad diaria: El volumen es muy consistente, entre 8.0 y 9.0 millones de posiciones por dia, con un promedio de 20,000 buques activos diariamente.

2. Perfilamiento de Embarcaciones y Dimensiones
+ Tipos de buque dominantes: El tipo 31 (Towing/Tug) y tipo 37 (Pleasure Craft) representan mas de 31 millones de posiciones. Los buques mercantes y tanques (Tipos 70 y 80) registran velocidades promedio sustancialmente mayores (5.8 a 6.3 nudos) en comparacion con embarcaciones menores (1.6 a 1.9 nudos).

+ Distribucion de tamano: La mediana de las embarcaciones presenta 22 metros de eslora (Length), 8 metros de manga (Width) y 3.7 metros de calado (Draft). En el percentil 99 se observan embarcaciones de hasta 323 metros de eslora y 17 metros de calado (portacontenedores y grandes tanqueros).

3. Diagnostico de Calidad de Datos (Insumo para Entrega 2)
+ Heading no disponible (511.0): Afecta a 33,535,628 registros (55.40%). Segun la norma ITU de AIS, el valor 511 no es un error corrupto, sino el codigo que emite el transceptor cuando el barco no tiene girocompas conectado o esta estatico.
+ Dimensiones invalidas: 2,284,033 registros (3.77%) tienen eslora o manga igual o menor a cero.
+ Velocidades imposibles: 160,613 registros (0.265%) reportan velocidades superiores a 60 nudos (SOG), causadas por picos de ruido en la señal GPS.
+ MMSI anomalos: 49,897 registros (0.082%) corresponden a identificadores invalidos (longitud diferente a 9 digitos o codigos de prueba como 000000000).
+ Coordenadas geograficas: Excelente calidad, no se registraron latitudes o longitudes fuera de rango global.

## Conclusion
El dataset presenta alta integridad geometrica en coordenadas, pero requiere reglas de limpieza claras para la Entrega 2: filtrar o imputar las dimensiones invalidas, tratar el valor 511 en Heading como categoria nula/no disponible en lugar de valor numerico, y descartar picos de velocidad superiores a 60 nudos.

# Requisito 3 Preguntas de negocio

#### a. ¿Cuántos buques distintos transmitieron cada día? Comparen el conteo exacto con approx_count_distinct y argumenten cuál usarían en producción.

In [0]:
unique_ships_day = (
    df_ais.withColumn("day", F.to_date("BaseDateTime"))
    .groupBy("day")
    .agg(F.countDistinct("MMSI").alias("unique_vessels"))
)
display(unique_ships_day)
unique_ships_day.explain("formatted")

day,unique_vessels
2023-06-04,19720
2023-06-05,19615
2023-06-02,21153
2023-06-06,19717
2023-06-01,20448
2023-06-07,20086
2023-06-03,20505


== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#11277, BaseDateTime#11278]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/noaa_data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSch

In [0]:
approx_unique_ships_day = (
    df_ais.withColumn("day", F.to_date("BaseDateTime"))
    .groupBy("day")
    .agg(F.approx_count_distinct("MMSI").alias("unique_vessels"))
)
display(approx_unique_ships_day)
approx_unique_ships_day.explain("formatted")

day,unique_vessels
2023-06-02,23182
2023-06-07,21184
2023-06-01,21782
2023-06-05,20358
2023-06-06,20288
2023-06-04,20275
2023-06-03,20877


== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonGroupingAgg (8)
         +- PhotonShuffleExchangeSource (7)
            +- PhotonShuffleMapStage (6)
               +- PhotonShuffleExchangeSink (5)
                  +- PhotonGroupingAgg (4)
                     +- PhotonProject (3)
                        +- PhotonRowToColumnar (2)
                           +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#11277, BaseDateTime#11278]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/noaa_data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<MMSI:string,BaseDateTime:timestamp>

(2) PhotonRowToColumnar
Input [2]: [MMSI#11277, BaseDateTime#11278]

(3) PhotonProject
Input [2]: [MMSI#11277, BaseDateTime#11278]
Arguments: [MMSI#11277, cast(BaseDateTime#11278 as date) AS day#12299]

(4) PhotonGroupingAgg
Input [2]: [MMSI#11277, day#12299]
Arguments: [day#12299],

El conteo difiere por poco para los dias. En produccion se usa el valor aproximado porque la importancia de este dato no justifica el precio de computo por un valor exacto. La aproximacion es suficiente para un reporte general.

#### b. ¿Qué tipos de buque generan más tráfico? Top 10 por número de posiciones, con velocidad media por tipo (usen el catálogo de tipos).

In [0]:
rank_vessel_type = (
    df_ais.groupBy("VesselType")
    .agg(
        F.count("*").alias("count"),
        F.avg("SOG").alias("average_speed")
    )
    .orderBy(F.desc("count"))
    .limit(10)
)

display(rank_vessel_type)
rank_vessel_type.explain("formatted")

VesselType,count,average_speed
31,16558120,1.8987526240930621
37,14476696,1.6422352103023863
60,4795209,4.433895873989091
30,4010188,2.466139193474545
36,3876264,1.9792365019505584
90,3848162,2.0281950188159845
70,3837926,6.369827792410587
52,1921565,2.093070023651827
80,1789168,5.874565719933229
57,1323765,1.9109420478708936


== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonTopK (12)
         +- PhotonShuffleExchangeSource (11)
            +- PhotonShuffleMapStage (10)
               +- PhotonShuffleExchangeSink (9)
                  +- PhotonTopK (8)
                     +- PhotonGroupingAgg (7)
                        +- PhotonShuffleExchangeSource (6)
                           +- PhotonShuffleMapStage (5)
                              +- PhotonShuffleExchangeSink (4)
                                 +- PhotonGroupingAgg (3)
                                    +- PhotonRowToColumnar (2)
                                       +- Scan csv  (1)


(1) Scan csv 
Output [2]: [SOG#11281, VesselType#11287]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/noaa_data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<SOG:double,VesselType:int>

(2) PhotonRowToColumnar
Input [2]: [SOG#1

**Respuesta.** Los tipos que generan más tráfico (por número de posiciones) son el **31 (Towing)**
con 16.5 M y el **37 (Pleasure Craft / remolcadores menores)** con 14.5 M; juntos concentran más de
la mitad de las emisiones. En velocidad media se ve el patrón esperado: los tipos de carga y tanqueros
—**70 (Cargo, 6.37 nudos)** y **80 (Tanker, 5.87 nudos)**— navegan sustancialmente más rápido que las
embarcaciones costeras y de servicio, que rondan 1.6–2.5 nudos. Es decir, el mayor **volumen** de
mensajes lo aportan embarcaciones lentas y muy numerosas, mientras que la mayor **velocidad** la
aportan los buques mercantes de línea, menos numerosos pero de tránsito continuo.

#### c. ¿Qué 10 buques recorrieron más distancia durante la semana? (orden por buque y tiempo, distancia entre posiciones consecutivas con haversine).

In [0]:
vessel_window = Window.partitionBy("MMSI").orderBy("BaseDateTime")
df = (
    df_ais.withColumn("prev_lat", F.lag("LAT").over(vessel_window))
    .withColumn("prev_lon", F.lag("LON").over(vessel_window))
)

lat1 = F.radians(F.col("prev_lat"))
lon1 = F.radians(F.col("prev_lon"))
lat2 = F.radians(F.col("LAT"))
lon2 = F.radians(F.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = F.sin(dlat / 2) ** 2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2

r_km = 6371.0

distance_km = F.when(
    F.col("prev_lat").isNotNull() & F.col("prev_lon").isNotNull(),
    2 * r_km * F.asin(F.sqrt(a))
).otherwise(0.0)

df = df.withColumn("distance", distance_km)

top_10_distance_vessels = (
    df.groupBy("MMSI")
    .agg(F.sum("distance").alias("total_distance_km"))
    .orderBy(F.desc("total_distance_km"))
    .limit(10)
)

display(top_10_distance_vessels)
top_10_distance_vessels.explain("formatted")


MMSI,total_distance_km
211002010,7931112.219051635
368124160,7361374.945480704
367373000,5724082.305732174
338333275,5065029.783610111
374158000,3027501.8058918915
338561000,1613419.3413086575
360000000,1155988.0831470503
338479554,852140.4119425962
367530470,317418.18156711175
338182085,249765.17639226164


== Physical Plan ==
AdaptiveSparkPlan (17)
+- == Initial Plan ==
   PhotonResultStage (16)
   +- PhotonColumnarToRow (15)
      +- PhotonTopK (14)
         +- PhotonShuffleExchangeSource (13)
            +- PhotonShuffleMapStage (12)
               +- PhotonShuffleExchangeSink (11)
                  +- PhotonTopK (10)
                     +- PhotonGroupingAgg (9)
                        +- PhotonProject (8)
                           +- PhotonWindow (7)
                              +- PhotonSort (6)
                                 +- PhotonShuffleExchangeSource (5)
                                    +- PhotonShuffleMapStage (4)
                                       +- PhotonShuffleExchangeSink (3)
                                          +- PhotonRowToColumnar (2)
                                             +- Scan csv  (1)


(1) Scan csv 
Output [4]: [MMSI#11258, BaseDateTime#11259, LAT#11260, LON#11261]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/de

##### Jusitificacion
**Puntos claves del plan**:
+ Column Pruning
+ hashpartitioning(MMSI) y ordenamiento local
+ Top-K eficiente

La consulta se hizo de esta manera porque usa expresiones nativas para calcular la formula de Haversine y se aplico Window.partitionBy("MMSI").orderBy("BaseDateTime") junto con un limit(10). El plan de ejecucion confirma que el nodo Scan csv aplico column pruning leyendo solo 4 de las 17 columnas (MMSI, BaseDateTime, LAT, LON), reduciendo la I/O desde el origen. El shuffle por `hashpartitioning(MMSI)` permite ordenar localmente los datos por tiempo para procesar el `lag()` con un marco de memoria pequeño, mientras que el nodo `PhotonTopK` filtra los 10 mejores registros en cada ejecutor antes de enviar datos por la red a SinglePartition. No se hizo con una UDF de Python ni trayendo todos los datos ordenados al driver porque habria generado una sobrecarga masiva de serializacion entre la JVM y Python, ademas de un cuello de botella de red al transferir millones de agregaciones intermedias sin filtrar previo al shuffle final.

#### d. ¿Dónde se concentra el tráfico? Top 10 celdas de una grilla espacial (H3 resolución 8, o su propia rejilla lat/lon) por número de posiciones, y cuáles de esas celdas corresponden a puertos del World Port Index.

In [0]:
# 1. Map each position to its H3 Resolution 8 cell ID
df_h3 = df_ais.withColumn(
    "h3_cell", 
    F.expr("h3_longlatash3string(LON, LAT, 8)")
)

# 2. Find Top 10 spatial cells by position count
top_10_cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(F.count("*").alias("position_count"))
    .orderBy(F.desc("position_count"))
    .limit(10)
)

display(top_10_cells)
top_10_cells.explain("formatted")

h3_cell,position_count
8828d555a1fffff,235268
8828d54715fffff,184414
8829a411d7fffff,180100
8828d17b59fffff,123785
8829a19a35fffff,115457
88446e0359fffff,114645
8829127827fffff,109522
8844a13b51fffff,103934
8828d5476bfffff,101039
8829a411a9fffff,100848


== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonTopK (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonTopK (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [LAT#16300, LON#16301]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<LAT:double,LON:do

In [0]:
# Leer el archivo de puertos del Wordl Port Index
df_wpi = (
    spark.read
    .option("header", "true")
    .csv("/Volumes/proyecto_datos/default/data/UpdatedPub150.csv")
    .select(
        F.col("World Port Index Number").cast("double"),
        F.col("Main Port Name"),
        F.col("Country Code"),
        F.col("Latitude").cast("double"),
        F.col("Longitude").cast("double")
    )
)
display(df_wpi.limit(10))

World Port Index Number,Main Port Name,Country Code,Latitude,Longitude
7950.0,Maurer,United States,40.533333,-74.25
47620.0,Iharana,Madagascar,-13.35,50.0
47020.0,Chake Chake,Tanzania,-5.25,39.766667
47005.0,Mjimwema Terminal,Tanzania,-6.816667,39.366667
44804.0,Delta Terminal,Turkey,36.85,36.166667
50350.0,Cinta Oil Terminal,Indonesia,-5.5,106.233333
38330.0,Europa Point,Gibraltar,36.133333,-5.35
7010.0,New Harbor,United States,43.866667,-69.483333
12900.0,Dtse / Gegua Oil Terminal,Brazil,-22.816667,-43.15
48104.0,Sharmah,Saudi Arabia,27.933333,35.25


In [0]:
# Calcular valor H3 para cada puerto
df_wpi_h3 = df_wpi.withColumn(
    "h3_cell",
    F.expr("h3_longlatash3string(Longitude, Latitude, 8)")
)

# Join Top 10 celdas AIS con World Port Index
top_10_ports_matched = (
    top_10_cells
    .join(df_wpi_h3, on="h3_cell", how="left")
    .select(
        "h3_cell",
        "position_count",
        F.col("Main Port Name").alias("matched_port_name"),
        F.col("Country Code").alias("country")
    )
    .orderBy(F.desc("position_count"))
)

display(top_10_ports_matched)
top_10_ports_matched.explain("formatted")

h3_cell,position_count,matched_port_name,country
8828d555a1fffff,235268,null,null
8828d54715fffff,184414,null,null
8829a411d7fffff,180100,null,null
8828d17b59fffff,123785,null,null
8829a19a35fffff,115457,null,null
88446e0359fffff,114645,null,null
8829127827fffff,109522,null,null
8844a13b51fffff,103934,null,null
8828d5476bfffff,101039,null,null
8829a411a9fffff,100848,null,null


== Physical Plan ==
AdaptiveSparkPlan (26)
+- == Initial Plan ==
   PhotonResultStage (25)
   +- PhotonColumnarToRow (24)
      +- PhotonSort (23)
         +- PhotonProject (22)
            +- PhotonBroadcastHashJoin LeftOuter (21)
               :- PhotonTopK (13)
               :  +- PhotonShuffleExchangeSource (12)
               :     +- PhotonShuffleMapStage (11)
               :        +- PhotonShuffleExchangeSink (10)
               :           +- PhotonTopK (9)
               :              +- PhotonGroupingAgg (8)
               :                 +- PhotonShuffleExchangeSource (7)
               :                    +- PhotonShuffleMapStage (6)
               :                       +- PhotonShuffleExchangeSink (5)
               :                          +- PhotonGroupingAgg (4)
               :                             +- PhotonProject (3)
               :                                +- PhotonRowToColumnar (2)
               :                                   +- Scan

Se encontro que para las posiciones mas transitadas no hay ningun puerto. El enlace directo por ID de celda fue demasiado estricto a esta resolución.

Esto podria explicarse en primer lugar porque la coordenada del puerto en el World Port Index es un punto arbitrario en tierra o sobre el muelle. Al asignarle una celda, esta no suele coincidir exactamente con la celda sobre el agua donde los barcos transmiten sus señales AIS. Ademas, a resolución 8, cada celda cubre solo ~0.73 km^2. Un barco transitando en la entrada del puerto estará a solo unos cientos de metros de la coordenada oficial, pero en una celda vecina.




## Justificación (d)

La consulta se hizo de esta manera porque el conteo de posiciones por zona es una **agregación por
clave** (`groupBy(h3_cell).count()`) de la que solo interesa el **Top 10**. El plan lo resuelve con
una agregación parcial en cada ejecutor (`PhotonGroupingAgg`), un único `hashpartitioning(h3_cell)`
y un `PhotonTopK` que recorta a 10 filas **antes** del `shuffle` final a `SinglePartition`, así que
por la red viajan decenas de filas y no millones. Se eligió la grilla **H3 resolución 8** en vez de
redondear lat/lon porque H3 da celdas de área casi constante (~0.73 km²) y vecindad regular; el mapeo
se hace con la expresión nativa `H3Utils.dbxH3LongLatAsH3String` (nodo `PhotonProject`), sin UDF de
Python, por lo que corre dentro de Photon sin costo de serialización JVM↔Python. El plan también
confirma *column pruning*: el `Scan csv` lee solo `[LAT, LON]`, 2 de las 17 columnas.

El cruce con el World Port Index se hizo con un **`join` por igualdad de `h3_cell`** (no un producto
cartesiano de distancias) porque comparar cada celda contra cada puerto sería O(celdas × puertos); al
llevar ambos lados a la misma clave H3, el `join` es una simple igualdad. El plan confirma que Spark
eligió un **`PhotonBroadcastHashJoin LeftOuter`** (`EXECUTOR_BROADCAST`): como el WPI es una tabla
pequeña, la difunde a cada ejecutor y evita un `shuffle` del lado grande (las Top-10 celdas).

**Resultado y su lectura:** el `join` exacto devolvió `null` para todas las celdas más transitadas.
No es un error del código sino una consecuencia de la resolución: a res 8 cada celda mide ~0.73 km²,
y la coordenada "oficial" de un puerto en el WPI es un punto en el muelle o en tierra, que cae en una
celda distinta a la que ocupan los barcos fondeados/transitando a unos cientos de metros. Es decir,
el emparejamiento por **ID de celda idéntico** es demasiado estricto para este propósito. No se
resolvió aquí con una búsqueda por distancia (vecindad `h3_grid_disk` o *range join* por radio)
porque eso corresponde al enriquecimiento de la Entrega 2; para la Entrega 1 basta con **cuantificar
la concentración de tráfico** (que sí se obtuvo) y documentar por qué el match directo no aplica.

#### e. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

In [0]:
vessel_days = (
    df_ais
    .withColumn("date", F.to_date("BaseDateTime"))
    .groupBy("MMSI")
    .agg(F.countDistinct("date").alias("days_active"))
)

metrics = vessel_days.select(
    F.count("*").alias("total_unique_vessels"),
    F.sum(F.when(F.col("days_active") == 7, 1).otherwise(0)).alias("vessels_7_days"),
    F.sum(F.when(F.col("days_active") == 1, 1).otherwise(0)).alias("vessels_1_day")
).collect()[0]

total_vessels = metrics["total_unique_vessels"]
count_7_days = metrics["vessels_7_days"]
count_1_day = metrics["vessels_1_day"]

prop_7_days = (count_7_days / total_vessels) * 100
prop_1_day = (count_1_day / total_vessels) * 100

print(f"Total Unique Vessels: {total_vessels:,}")
print(f"Vessels active all 7 days: {count_7_days:,} ({prop_7_days:.2f}%)")
print(f"Single-day visitors (1 day only): {count_1_day:,} ({prop_1_day:.2f}%)")

vessel_days.explain("formatted")


Total Unique Vessels: 31,871
Vessels active all 7 days: 12,667 (39.74%)
Single-day visitors (1 day only): 5,996 (18.81%)
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#16298, BaseDateTime#16299]
Batched: false
Locati

El 39.74% del total de la flota, emitio señales durante los siete dias. El 18.81%, fueron visitantes de un solo dia.

In [0]:
# Filtrar MMSIs de un solo dia
single_day_mmsis = vessel_days.filter(F.col("days_active") == 1).select("MMSI")

# Mapear visitantes de un solo dia con su ubicacion espacial (H3 resolucion 8)
single_day_locations = (
    df_ais
    .join(single_day_mmsis, on="MMSI", how="inner")
    .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        F.count("*").alias("position_count"),
        F.countDistinct("MMSI").alias("unique_single_day_vessels"),
    )
    .orderBy(F.desc("unique_single_day_vessels"))
)

display(single_day_locations.limit(10))
single_day_locations.explain("formatted")

h3_cell,position_count,unique_single_day_vessels
882aa802cdfffff,266,49
8828d54715fffff,1319,47
882aa80213fffff,239,43
882aa80217fffff,811,42
8844a13b2dfffff,325,41
882aa80211fffff,273,41
882aa8021bfffff,245,39
8844a13b67fffff,179,38
882aa802e9fffff,510,35
8828d54711fffff,132,34


== Physical Plan ==
AdaptiveSparkPlan (41)
+- == Initial Plan ==
   PhotonResultStage (40)
   +- PhotonColumnarToRow (39)
      +- PhotonSort (38)
         +- PhotonShuffleExchangeSource (37)
            +- PhotonShuffleMapStage (36)
               +- PhotonShuffleExchangeSink (35)
                  +- PhotonGroupingAgg (34)
                     +- PhotonShuffleExchangeSource (33)
                        +- PhotonShuffleMapStage (32)
                           +- PhotonShuffleExchangeSink (31)
                              +- PhotonGroupingAgg (30)
                                 +- PhotonGroupingAgg (29)
                                    +- PhotonGroupingAgg (28)
                                       +- PhotonProject (27)
                                          +- PhotonShuffledHashJoin Inner (26)
                                             :- PhotonShuffleExchangeSource (6)
                                             :  +- PhotonShuffleMapStage (5)
                           

Respecto a la ubicacion espacial de los visitantes de un solo dia, las celdas revelan que la mayor concentracion se registra en la celda `882aa802cdfffff` con 49 embarcaciones distintas, seguida por la celda `8828d54715fffff` que registra 47 buques unicos.

## Justificación (e)

**Proporción de buques por días activos.** Se calculó con `groupBy(MMSI).agg(countDistinct(date))`
para obtener los días distintos en que emitió cada buque, y luego una sola pasada de agregación
condicional (`sum(when(days_active == 7))` y `== 1`) para contar los dos extremos. El plan muestra la
agregación en dos niveles por `MMSI` con `hashpartitioning(MMSI)`: primero se distinguen los pares
`(MMSI, date)` y después se cuentan por buque, todo en Spark nativo. No se usó una `Window` con
`collect_set` de fechas porque materializar el conjunto de fechas por buque es más caro en memoria que
un `countDistinct`, y aquí solo interesa el **número** de días, no el conjunto.

**Dónde están los visitantes de un solo día.** Se aisló la lista de MMSI con `days_active == 1`
(`PhotonFilter days_active = 1`) y se volvió a unir contra `df_ais` (`how="inner"`) para geolocalizar
solo esas emisiones y agregarlas por celda H3. El plan resuelve el cruce con un
**`PhotonShuffledHashJoin Inner`**: ambos lados se reparticionan por `hashpartitioning(MMSI)` y se
unen por clave. Es un *inner join* que actúa como semi-join —descarta de entrada las posiciones de
buques con más de un día— antes del `groupBy(h3_cell)`, así que al agregado espacial solo llegan las
filas relevantes.

> **Sobre el tipo de `join`:** aquí Spark **no** difundió la lista de MMSI (no aparece
> `BroadcastHashJoin`) sino que hizo un `ShuffledHashJoin`. Es esperable: el lado pequeño no es una
> tabla con tamaño conocido, sino la **salida de una cadena de agregaciones** (`countDistinct` +
> `filter`), por lo que el optimizador no dispone de una estadística de tamaño fiable en tiempo de
> plan y opta por reparticionar ambos lados. Si se quisiera forzar la difusión —y ahorrar el shuffle
> del lado grande cuando sabemos que la lista es pequeña (~6.000 MMSI)— bastaría envolver el lado
> pequeño en `F.broadcast(single_day_mmsis)`. No se hizo con `isin()` de miles de valores ni con una
> subconsulta correlacionada porque el `join` por clave es el patrón que Spark optimiza para filtrar
> una tabla grande con una lista derivada.

# Requisito 4

## Propósito de consulta y decisiones de almacenamiento

**Propósito de consulta (concreto):** *el monitoreo diario del operador portuario*, que
consulta todas las posiciones de un **día específico** (`date`) dentro de un **sector marítimo**
delimitado por una celda de la grilla espacial (`h3_cell`, resolución 8). Es decir, el patrón de
acceso dominante es un filtro `WHERE date = :d AND h3_cell = :c`.

**Decisiones (se justifican con la evidencia de los pasos B–F):**

| Decisión | Elección | Por qué |
|---|---|---|
| **Formato de archivo** | **Parquet** (columnar, comprimido) sobre CSV | CSV es texto sin tipado ni compresión: para leer 4 columnas hay que parsear la fila completa. Parquet guarda por columna con estadísticas min/max por bloque, habilitando *column pruning* y *predicate pushdown*. |
| **Formato de tabla** | **Delta** sobre Parquet "a secas" | Delta = Parquet + un *transaction log*. Ese log guarda estadísticas por archivo y permite **data skipping**, `OPTIMIZE` (compactación) y `ZORDER` (co-localización). Parquet solo no tiene log, así que no puede saltarse archivos por estadísticas ni compactar de forma transaccional. |
| **Layout — particionamiento** | `partitionBy("date")` | El propósito **siempre** filtra por fecha. Particionar por `date` crea 7 carpetas físicas; un filtro por fecha lee solo 1 (*partition pruning*), descartando 6/7 de los datos sin abrirlos. |
| **Layout — co-localización** | `OPTIMIZE ... ZORDER BY (h3_cell)` | Dentro de cada partición de fecha, el `ZORDER` agrupa las filas de una misma celda espacial en los mismos archivos. Con las estadísticas min/max del log, Delta salta los archivos que no contienen la celda buscada (*data skipping*), la segunda mitad del filtro del propósito. |

No se elige CSV como formato final porque no permite *pushdown* ni *skipping* (hay que leer y parsear todo).<br>
No se elige Parquet suelto como tabla porque, sin el log de Delta, no hay estadísticas por archivo para saltar datos, ni `OPTIMIZE`/`ZORDER`. La evidencia de más abajo cuantifica ambas cosas.

## Paso A — Preparar columnas de partición (`date`) y grilla espacial (`h3_cell`)

Sobre el DataFrame de posiciones de los 7 días (`df_ais`) derivamos las dos columnas que usa el propósito.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, countDistinct, input_file_name

# Rutas base
CSV_PATH     = "/Volumes/proyecto_datos/default/data/ais_data/"
PARQUET_PATH = "/Volumes/proyecto_datos/default/data/ais_parquet/"
DELTA_TABLE  = "proyecto_datos.default.ais_positions_delta"

# date (partición) + h3_cell (sector espacial del propósito)
df_prepared = (
    df_ais
    .withColumn("date", to_date(col("BaseDateTime")))
    .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
)

display(df_prepared.select("MMSI", "BaseDateTime", "date", "h3_cell").limit(5))

MMSI,BaseDateTime,date,h3_cell
368069540,2023-06-02T00:00:00.000Z,2023-06-02,8828c3c8a9fffff
538004494,2023-06-02T00:00:01.000Z,2023-06-02,88282235c9fffff
367614020,2023-06-02T00:00:01.000Z,2023-06-02,8828195807fffff
367372810,2023-06-02T00:00:02.000Z,2023-06-02,884450c5c1fffff
367148660,2023-06-02T00:00:05.000Z,2023-06-02,88464bb227fffff


## Paso B — Escribir las dos alternativas: Parquet y Delta

Escribimos **el mismo DataFrame** en los dos formatos que vamos a comparar, ambos **particionados por `date`**
para que la comparación sea justa (mismo layout lógico, distinto motor de tabla).

In [0]:
# --- Baseline en Parquet (archivo columnar, sin transaction log) ---
(df_prepared.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("date")
    .save(PARQUET_PATH)
)

# --- Tabla Delta (Parquet + transaction log) particionada por fecha ---
(df_prepared.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("date")
    .option("overwriteSchema", "true")
    .saveAsTable(DELTA_TABLE)
)

print("Parquet escrito en:", PARQUET_PATH)
print("Tabla Delta creada:", DELTA_TABLE)

Parquet escrito en: /Volumes/proyecto_datos/default/data/ais_parquet/
Tabla Delta creada: proyecto_datos.default.ais_positions_delta


## Paso C — Evidencia 1: bytes antes / después (CSV → Parquet → Delta)

Medimos el tamaño **real** de cada representación. Para CSV y Parquet recorremos el directorio con
`dbutils.fs`; para Delta leemos el tamaño autoritativo del *transaction log* con `DESCRIBE DETAIL`
(así no medimos una carpeta equivocada).

In [0]:
def dir_size_bytes(path):
    """Suma recursiva del tamaño de todos los archivos bajo un path del Volume."""
    total = 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            total += dir_size_bytes(f.path)
        else:
            total += f.size
    return total

def gb(b):
    return b / (1024 ** 3)

# CSV y Parquet: tamaño en disco por recorrido de directorio
size_csv     = dir_size_bytes(CSV_PATH)
size_parquet = dir_size_bytes(PARQUET_PATH)

# Delta: tamaño autoritativo desde el log (sizeInBytes) + numFiles
detail = spark.sql(f"DESCRIBE DETAIL {DELTA_TABLE}").select("sizeInBytes", "numFiles").collect()[0]
size_delta      = detail["sizeInBytes"]
delta_num_files = detail["numFiles"]

print(f"CSV     : {gb(size_csv):6.2f} GB")
print(f"Parquet : {gb(size_parquet):6.2f} GB   ({(1 - size_parquet/size_csv)*100:5.1f}% menos que CSV)")
print(f"Delta   : {gb(size_delta):6.2f} GB   ({(1 - size_delta/size_csv)*100:5.1f}% menos que CSV)  | archivos: {delta_num_files}")

CSV     :   6.04 GB
Parquet :   2.01 GB   ( 66.7% menos que CSV)
Delta   :   1.48 GB   ( 75.6% menos que CSV)  | archivos: 7


## Paso D — Evidencia 2: archivos escaneados por la consulta del propósito

Ejecutamos la consulta del propósito (`date = :d AND h3_cell = :c`) sobre cada fuente y
recogemos **dos evidencias complementarias**:

1. **Plan de ejecución** (`explain("formatted")`): en el nodo `Scan` se leen los
   `PartitionFilters` (prueba del *partition pruning* por `date`) y las `DictionaryFilters` /
   `RequiredDataFilters` sobre `h3_cell` (el *data skipping* que Delta aplica con las
   estadísticas de su *transaction log*).
2. **Archivos con datos** (`_metadata.file_path`): número de archivos físicos que aportan filas al
   resultado. Menos archivos = mejor *pruning/skipping* para este patrón de acceso.
   *(Nota: en Unity Catalog `input_file_name()` no está soportado; se usa la columna
   `_metadata.file_path`.)*

> Ambos formatos aplican el mismo *partition pruning* por `date` (idéntico `PartitionFilters` en el
> plan), así que descartan 6 de los 7 días. La diferencia aparece **dentro** del día: Parquet debe
> abrir todos los *part-files* de esa partición, mientras que Delta usa las estadísticas por archivo
> de su log para saltar los que no contienen la celda buscada.

In [0]:
target_date = "2023-06-01"
target_cell = "882aa802cdfffff"   # celda con más visitantes de un solo día (Req 3e)

def purpose_query(df):
    """La consulta del propósito: filtra por fecha (partición) y sector espacial (h3_cell)."""
    return df.filter((col("date") == target_date) & (col("h3_cell") == target_cell))

def files_and_rows(df, label, show_plan=False):
    q = purpose_query(df)
    n_files = q.withColumn("_file", col("_metadata.file_path")).select(countDistinct("_file")).collect()[0][0]
    n_rows  = q.count()
    print(f"{label:26s} -> archivos con datos: {n_files:4d} | filas: {n_rows}")
    if show_plan:
        print(f"\n----- PLAN: {label} -----")
        q.explain("formatted")
        print()
    return n_files, n_rows

df_parquet = spark.read.format("parquet").load(PARQUET_PATH)
df_delta   = spark.read.table(DELTA_TABLE)

print("=== Archivos escaneados por la consulta del propósito ===")
f_par, _     = files_and_rows(df_parquet, "Parquet (part. date)",   show_plan=True)
f_del_pre, _ = files_and_rows(df_delta,   "Delta ANTES de OPTIMIZE", show_plan=True)

=== Archivos escaneados por la consulta del propósito ===
Parquet (part. date)       -> archivos con datos:    6 | filas: 242

----- PLAN: Parquet (part. date) -----
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [19]: [MMSI#14874, BaseDateTime#14875, LAT#14876, LON#14877, SOG#14878, COG#14879, Heading#14880, VesselName#14881, IMO#14882, CallSign#14883, VesselType#14884, Status#14885, Length#14886, Width#14887, Draft#14888, Cargo#14889, TransceiverClass#14890, h3_cell#14891, date#14892]
DictionaryFilters: [(h3_cell#14891 = 882aa802cdfffff)]
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_parquet]
PartitionFilters: [isnotnull(date#14892), (date#14892 = 2023-06-01)]
ReadSchema: struct<MMSI:string,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double,COG:double,Heading:double,VesselName:string,IMO:string,CallSign:string,VesselType:int,Status:int,Length:double,Width:double,Draf

## Paso E — Aplicar `OPTIMIZE` + `ZORDER BY (h3_cell)`

`OPTIMIZE` reescribe los datos de cada partición y `ZORDER BY (h3_cell)` co-localiza en los mismos
archivos las filas de una misma celda espacial, para maximizar el *data skipping* de consultas por
zona. La fila de métricas de salida (`numFilesAdded` / `numFilesRemoved`) documenta la reescritura.

> **Qué esperar a esta escala.** Nuestra tabla es pequeña por partición (~7 archivos, uno por día).
> `OPTIMIZE` no *reduce* el número de archivos aquí: al aplicar `ZORDER` la tabla pasa de **7 a 21
> archivos**, porque el reordenamiento por celda divide cada archivo diario en varios *Z-cubes*. La
> compactación que baja el conteo de archivos solo ayuda cuando hay **muchos archivos pequeños por
> partición** (p. ej. escritura por *streaming* o *merge* incremental), que no es nuestro caso. Por
> eso medimos el efecto sobre la **consulta**, no solo sobre el conteo de archivos.

In [0]:
%sql
-- Compactación + co-localización espacial dentro de cada partición de fecha
OPTIMIZE proyecto_datos.default.ais_positions_delta
ZORDER BY (h3_cell);

path,metrics
,"List(21, 7, List(62415407, 83490662, 7.238044409523809E7, 21, 1519989326), List(210409463, 240171464, 2.2631319057142857E8, 7, 1584192334), 7, List(minCubeSize(107374182400), List(0, 0), List(7, 1584192334), 0, List(7, 1584192334), 7, null), null, 0, 1, 7, 0, false, 0, 0, 1790365732491, 1790365808310, 8, 7, null, List(0, 0), null, 19, 19, 129333, 0, null, null, 0)"


In [0]:
# Volver a medir tras OPTIMIZE/ZORDER (importante: releer la tabla)
df_delta_opt = spark.read.table(DELTA_TABLE)

print("=== Archivos escaneados: efecto de OPTIMIZE + ZORDER ===")
f_del_post, _ = files_and_rows(df_delta_opt, "Delta DESPUÉS de OPTIMIZE", show_plan=True)

detail2 = spark.sql(f"DESCRIBE DETAIL {DELTA_TABLE}").select("numFiles", "sizeInBytes").collect()[0]
print()
print(f"Delta numFiles: {delta_num_files} (antes)  ->  {detail2['numFiles']} (después de compactar)")
print("Archivos tocados por la consulta del propósito:")
print(f"  Parquet                 : {f_par}")
print(f"  Delta (antes OPTIMIZE)  : {f_del_pre}")
print(f"  Delta (después ZORDER)  : {f_del_post}")

=== Archivos escaneados: efecto de OPTIMIZE + ZORDER ===
Delta DESPUÉS de OPTIMIZE  -> archivos con datos:    1 | filas: 242

----- PLAN: Delta DESPUÉS de OPTIMIZE -----
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet proyecto_datos.default.ais_positions_delta (1)


(1) PhotonScan parquet proyecto_datos.default.ais_positions_delta
Output [19]: [MMSI#16067, BaseDateTime#16068, LAT#16069, LON#16070, SOG#16071, COG#16072, Heading#16073, VesselName#16074, IMO#16075, CallSign#16076, VesselType#16077, Status#16078, Length#16079, Width#16080, Draft#16081, Cargo#16082, TransceiverClass#16083, h3_cell#16085, date#16084]
DictionaryFilters: [(h3_cell#16085 = 882aa802cdfffff)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-f010s/uc/6ddea1a1-9973-40df-a29d-eb4dbd4bf962/4735a373-8e6d-431c-8096-841dd3b90846/__unitystorage/catalogs/a4f6ad21-10a4-4747-a7e5-a3692caa2e16/tables/cae1f65b-ff43-4b0d-92de-4ca97ca2647c]
PartitionF

## Resumen comparativo (formato · bytes · archivos leídos)

Tabla única con las tres evidencias juntas para la consulta del propósito
(`date = 2023-06-01 AND h3_cell = 882aa802cdfffff`).

In [0]:
# Resumen: consolida bytes (Paso C) y archivos leídos por la consulta (Pasos D y E)
resumen = spark.createDataFrame(
    [
        ("CSV (crudo)",              round(gb(size_csv), 2),     None,       None),
        ("Parquet (part. date)",     round(gb(size_parquet), 2), int(f_par), None),
        ("Delta (antes OPTIMIZE)",   round(gb(size_delta), 2),   int(f_del_pre),  int(delta_num_files)),
        ("Delta (después ZORDER)",   round(gb(size_delta), 2),   int(f_del_post), int(detail2["numFiles"])),
    ],
    ["formato", "tamano_GB", "archivos_leidos_consulta", "archivos_totales_tabla"],
)

display(resumen)

formato,tamano_GB,archivos_leidos_consulta,archivos_totales_tabla
CSV (crudo),6.04,null,null
Parquet (part. date),2.01,6,null
Delta (antes OPTIMIZE),1.48,1,7
Delta (después ZORDER),1.48,1,21


## Conclusión — justificación con evidencia

El almacenamiento se resolvió con **Delta particionado por `date`** porque el propósito de consulta
filtra siempre por fecha y por sector espacial, y el layout ataca esas dos dimensiones. La evidencia
medida lo respalda:

1. **Bytes (Paso C):** la compresión columnar reduce el tamaño de forma drástica frente al CSV crudo:
   **CSV 6.04 GB → Parquet 2.01 GB (−66.7%) → Delta 1.48 GB (−75.6%)**.
2. **Archivos leídos por la consulta del propósito (Paso D):** con el mismo `PartitionFilters` por
   `date`, **Parquet abre 6 archivos** de la partición del día, mientras que **Delta abre solo 1**.
   Esa diferencia (6 → 1) es el *data skipping* de Delta: las estadísticas por archivo de su
   *transaction log*, combinadas con las `DictionaryFilters`/`RequiredDataFilters` sobre `h3_cell`
   del plan, le permiten saltar los archivos que no contienen la celda buscada. Es la ganancia de
   Delta sobre Parquet, y ocurre **sin necesidad de `OPTIMIZE`**.
3. **Efecto de `OPTIMIZE` + `ZORDER` (Paso E):** a nuestra escala **no mejora la consulta** (Delta ya
   leía 1 solo archivo por partición) y de hecho **sube el número de archivos de 7 a 21** por el
   reordenamiento en *Z-cubes*. `OPTIMIZE` es una herramienta para tablas con muchos archivos
   pequeños por partición; con 7 días y un archivo por día no hay nada que compactar, así que su
   beneficio es marginal aquí. Lo dejamos documentado como parte del análisis, no como una mejora
   observada.

**Decisión final:** *Delta particionado por `date`*. No se usa **CSV** como formato final porque es
texto sin tipado ni estadísticas (obliga a parsear la fila completa, sin *pushdown* ni *skipping*).
No se usa **Parquet suelto** porque, sin el *transaction log* de Delta, no tiene estadísticas por
archivo y por eso abre 6 archivos donde Delta abre 1. `ZORDER`/`OPTIMIZE` se mantienen en el pipeline
porque **escalarían** con más días o escritura incremental, aunque en este corpus de 7 días su efecto
sobre la consulta sea nulo.